In [1]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [2]:
train = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')
test = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv')

In [3]:
for i in zip(train.columns, train.dtypes):
    train[i[0]].fillna(0, inplace=True)

In [4]:
train_split, val_split = train_test_split(
    train.drop(columns=["date_id"]), test_size=0.01, random_state=42
)

X_train = train_split.drop(columns=["forward_returns"])
X_test = val_split.drop(columns=["forward_returns"])
y_train = train_split['forward_returns']
y_test = val_split['forward_returns']

In [5]:
X_train

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V2,V3,V4,V5,V6,V7,V8,V9,risk_free_rate,market_forward_excess_returns
7981,0,0,0,0,0,0,0,0,1,1.610946,...,0.958995,0.803571,0.952381,2.500989,0.302249,-0.185849,0.781746,0.057418,1.587302e-06,-0.003884
2648,0,0,0,0,0,-1,0,0,0,2.145950,...,0.463624,0.318122,0.388889,-0.844654,0.000661,0.370983,0.000661,0.000000,2.257937e-04,0.012340
879,0,0,0,0,0,-1,0,0,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.238095e-04,0.008005
5969,0,0,0,0,0,0,0,0,0,1.006096,...,0.656746,0.722222,0.698413,-0.088484,0.904101,-0.800329,0.181217,-0.955378,3.968254e-07,0.007065
766,0,0,0,1,0,0,0,0,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.198413e-04,0.004309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,0,0,0,0,1,0,0,1,0,1.126288,...,0.030423,0.012566,0.065476,1.651280,0.712302,-0.766849,0.958333,-0.810129,3.571429e-06,0.002294
5191,0,0,0,0,0,0,1,0,0,2.964092,...,0.491402,0.461640,0.404101,0.413520,0.927910,0.076761,0.599206,0.415710,5.555556e-06,-0.004371
5390,0,0,0,0,0,-1,0,0,0,2.060774,...,0.401455,0.293651,0.434524,0.308873,0.709656,-0.560071,0.730820,-0.453336,1.587302e-06,-0.008254
860,0,0,0,0,0,0,0,1,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.210317e-04,-0.003859


In [6]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor), 
              ('ElasticNet', ElasticNetCV()), ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
              ('LassoCV', LassoCV()), ('LassoLars', LassoLars())]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

StackingRegressor(cv=3,
                  estimators=[('CatBoost',
                               <catboost.core.CatBoostRegressor object at 0x7c9328fd3d50>),
                              ('XGBoost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.7, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,
                                            feature_types=None, gamma=None,
                                            g...
                                                   min_samples_split=5,
                                                   random_state=42)),
                              ('GBRegressor',
                               GradientBoostingRegressor(max_depth=8,
                                                         max_features='sqrt',
                                                         min_samples_leaf=50,
                                                         min_samples_split=500,
                                                         random_state=10,
                                                         subsample=0.8)),
                              ('ElasticNet', ElasticNetCV()),
                              ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
                              ('LassoCV', LassoCV()),
                              ('LassoLars', LassoLars())],
                  final_estimator=RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))

In [7]:
X_train

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V2,V3,V4,V5,V6,V7,V8,V9,risk_free_rate,market_forward_excess_returns
7981,0,0,0,0,0,0,0,0,1,1.610946,...,0.958995,0.803571,0.952381,2.500989,0.302249,-0.185849,0.781746,0.057418,1.587302e-06,-0.003884
2648,0,0,0,0,0,-1,0,0,0,2.145950,...,0.463624,0.318122,0.388889,-0.844654,0.000661,0.370983,0.000661,0.000000,2.257937e-04,0.012340
879,0,0,0,0,0,-1,0,0,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.238095e-04,0.008005
5969,0,0,0,0,0,0,0,0,0,1.006096,...,0.656746,0.722222,0.698413,-0.088484,0.904101,-0.800329,0.181217,-0.955378,3.968254e-07,0.007065
766,0,0,0,1,0,0,0,0,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.198413e-04,0.004309
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5734,0,0,0,0,1,0,0,1,0,1.126288,...,0.030423,0.012566,0.065476,1.651280,0.712302,-0.766849,0.958333,-0.810129,3.571429e-06,0.002294
5191,0,0,0,0,0,0,1,0,0,2.964092,...,0.491402,0.461640,0.404101,0.413520,0.927910,0.076761,0.599206,0.415710,5.555556e-06,-0.004371
5390,0,0,0,0,0,-1,0,0,0,2.060774,...,0.401455,0.293651,0.434524,0.308873,0.709656,-0.560071,0.730820,-0.453336,1.587302e-06,-0.008254
860,0,0,0,0,0,0,0,1,0,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.210317e-04,-0.003859


In [8]:
def predict(test: pl.DataFrame) -> float:
    test = test.rename({ 'lagged_risk_free_rate' : 'risk_free_rate', 
                        'lagged_market_forward_excess_returns':'market_forward_excess_returns'}).to_pandas().drop(columns=["lagged_forward_returns", 
                                                                                                                           "date_id", "is_scored"])
    for i in zip(test.columns, test.dtypes):
        test[i[0]].fillna(0, inplace=True)
    raw_pred = model.predict(test)[0]
    return raw_pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))